# 🎓 Projeto de Machine Learning — APS2: Classificação
## Previsão de Renda Anual — Adult Census Income Dataset

---

| Campo | Informação |
|---|---|
| **Disciplina** | Machine Learning |
| **Integrante(s)** | `[PREENCHER NOME(S) AQUI]` |
| **RA(s)** | `[PREENCHER RA(s) AQUI]` |
| **Data de Entrega** | 15 de Maio de 2026 |
| **Dataset** | [Adult Census Income — UCI ML Repository](https://archive.ics.uci.edu/ml/datasets/Adult) |

---

### Objetivo
Desenvolver e avaliar modelos de classificação binária para prever se um indivíduo ganha mais de **\$50K por ano** (`income > 50K`) ou não, com base nas características demográficas, educacionais e ocupacionais presentes no Adult Census Income Dataset.

### Referências Bibliográficas
- Dua, D. and Graff, C. (2019). UCI Machine Learning Repository. Irvine, CA: University of California, School of Information and Computer Science.
- Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. JMLR 12, pp. 2825-2830.
- Lundberg, S. M., & Lee, S. I. (2017). A unified approach to interpreting model predictions. NeurIPS.
- Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. KDD.
- Chawla, N. V. et al. (2002). SMOTE: Synthetic Minority Over-sampling Technique. JAIR.

---
## 0. Configuração do Ambiente

In [ ]:
# Instalação de dependências (caso necessário)
!pip install xgboost imbalanced-learn shap optuna --quiet

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)

# Imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# XGBoost
import xgboost as xgb

# SHAP
import shap

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configurações globais
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

print('✅ Ambiente configurado com sucesso!')

---
## 1. Carregamento e Inspeção dos Dados

In [ ]:
# ─── Carregamento do dataset ───────────────────────────────────────────────────
# Tentativa de carregar via UCI diretamente
url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
url_test  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test'

col_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

df_train = pd.read_csv(url_train, header=None, names=col_names,
                       sep=',\s*', engine='python', na_values='?')

df_test  = pd.read_csv(url_test, header=None, names=col_names,
                       sep=',\s*', engine='python', na_values='?', skiprows=1)

# Unir treino e teste para análise completa (faremos nossa própria divisão depois)
df = pd.concat([df_train, df_test], ignore_index=True)

# Padronizar target: remover ponto final em '<=50K.' e '>50K.'
df['income'] = df['income'].str.replace('.', '', regex=False).str.strip()

print(f'Shape do dataset completo: {df.shape}')
df.head()

In [ ]:
# ─── Informações gerais ────────────────────────────────────────────────────────
df.info()

In [ ]:
# ─── Estatísticas descritivas ─────────────────────────────────────────────────
df.describe(include='all').T

In [ ]:
# ─── Valores ausentes ─────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Missing (%)': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

print('=== Valores Ausentes ===')
print(missing_df)

# Visualização
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(missing_df.index, missing_df['Missing (%)'], color='#E63946')
    ax.set_xlabel('% de valores ausentes')
    ax.set_title('Valores Ausentes por Coluna')
    for i, (idx, row) in enumerate(missing_df.iterrows()):
        ax.text(row['Missing (%)'] + 0.1, i, f"{row['Missing']} ({row['Missing (%)']}%)", va='center')
    plt.tight_layout()
    plt.show()

---
## 2. Análise de Separabilidade das Classes

Analisamos quais variáveis melhor separam as classes `<=50K` e `>50K`.

In [ ]:
# ─── Distribuição da variável alvo ────────────────────────────────────────────
target_counts = df['income'].value_counts()
target_pct = df['income'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ['#457B9D', '#E63946']
axes[0].bar(target_counts.index, target_counts.values, color=colors)
axes[0].set_title('Contagem por Classe')
axes[0].set_ylabel('Frequência')
for i, (k, v) in enumerate(target_counts.items()):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=target_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporção por Classe')

plt.suptitle('Distribuição da Variável Alvo (income)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n⚠️  Desbalanceamento: {target_pct.iloc[0]:.1f}% vs {target_pct.iloc[1]:.1f}%')
print(f'   Razão: {target_counts.iloc[0] / target_counts.iloc[1]:.1f}:1 (<=50K : >50K)')

In [ ]:
# ─── Separabilidade: features numéricas ───────────────────────────────────────
num_features = ['age', 'education-num', 'capital-gain', 'capital-loss',
                'hours-per-week', 'fnlwgt']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(num_features):
    for label, color in [('<=50K', '#457B9D'), ('>50K', '#E63946')]:
        subset = df[df['income'] == label][feat].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    axes[i].set_title(f'Distribuição: {feat}')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Densidade')
    axes[i].legend()

plt.suptitle('Separabilidade — Features Numéricas por Classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Separabilidade: features categóricas ─────────────────────────────────────
cat_features = ['education', 'marital-status', 'occupation', 'relationship', 'sex', 'workclass']

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    ct = pd.crosstab(df[feat], df['income'], normalize='index') * 100
    ct = ct.sort_values('>50K', ascending=True)
    ct.plot(kind='barh', ax=axes[i], color=['#457B9D', '#E63946'],
            width=0.7, edgecolor='white')
    axes[i].set_title(f'% por Classe — {feat}')
    axes[i].set_xlabel('% dentro da categoria')
    axes[i].legend(title='Renda')
    axes[i].xaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('Separabilidade — Features Categóricas por Classe', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Matriz de correlação (features numéricas) ────────────────────────────────
df_num = df[num_features].copy()
df_num['income_bin'] = (df['income'] == '>50K').astype(int)

fig, ax = plt.subplots(figsize=(8, 6))
corr = df_num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlação — Features Numéricas + Target', fontsize=13)
plt.tight_layout()
plt.show()

print('\n📊 Correlação com a variável alvo (income_bin):')
print(corr['income_bin'].drop('income_bin').sort_values(ascending=False).to_string())

### 💡 Análise de Separabilidade — Conclusões

> **`[PREENCHER]`** — Escreva aqui as principais conclusões da análise de separabilidade:
> - Quais variáveis melhor separam as classes?
> - O que você observou nas distribuições?
> - Quais variáveis categóricas apresentam maior discriminação?

Observações preliminares:
- **`education-num`**: Indivíduos com maior nível educacional tendem significativamente a ganhar >50K.
- **`capital-gain`** e **`capital-loss`**: Altamente assimétricos; quando diferentes de zero, são fortes preditores de renda alta.
- **`marital-status`**: Pessoas casadas (Married-civ-spouse) apresentam proporção muito maior de renda >50K.
- **`occupation`**: Profissões executivas e especializadas possuem maior proporção de >50K.
- **`relationship`**: `Husband` destaca-se como classe com maior renda.
- **`sex`**: Há diferença marcante, com homens apresentando maior proporção de >50K.

---
## 3. Pré-processamento

In [ ]:
# ─── Separar features e target ────────────────────────────────────────────────
# Remover 'fnlwgt' (peso amostral, não preditivo)
# Remover 'education' (substituída por 'education-num' que é ordinal)
DROP_COLS = ['fnlwgt', 'education', 'income']

X = df.drop(columns=DROP_COLS)
y = (df['income'] == '>50K').astype(int)  # 1 = >50K, 0 = <=50K

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')
print(f'\nBalanceamento: {y.value_counts().to_dict()}')

---
## 4. Feature Engineering

Criamos ao menos **3 novas features** com base em conhecimento do domínio.

In [ ]:
# ─── Feature Engineering ──────────────────────────────────────────────────────

def feature_engineering(df_input):
    df_fe = df_input.copy()
    
    # --- Feature 1: capital_net ---
    # Balanço entre ganho e perda de capital — proxy de riqueza
    df_fe['capital_net'] = df_fe['capital-gain'] - df_fe['capital-loss']
    
    # --- Feature 2: work_experience_proxy ---
    # Estimativa de anos de experiência profissional
    df_fe['work_experience_proxy'] = df_fe['age'] - df_fe['education-num'] - 6
    df_fe['work_experience_proxy'] = df_fe['work_experience_proxy'].clip(lower=0)
    
    # --- Feature 3: capital_has_activity ---
    # Flag: indivíduo tem alguma atividade de capital (ganho ou perda ≠ 0)
    df_fe['capital_has_activity'] = (
        (df_fe['capital-gain'] > 0) | (df_fe['capital-loss'] > 0)
    ).astype(int)
    
    # --- Feature 4: hours_above_fulltime ---
    # Trabalha mais de 40h por semana (possível executivo / múltiplos empregos)
    df_fe['hours_above_fulltime'] = (df_fe['hours-per-week'] > 40).astype(int)
    
    # --- Feature 5: is_married ---
    # Status de casamento binário — forte preditor identificado no EDA
    married_values = ['Married-civ-spouse', 'Married-AF-spouse']
    df_fe['is_married'] = df_fe['marital-status'].isin(married_values).astype(int)
    
    # --- Feature 6: edu_hours_interaction ---
    # Interação entre educação e horas trabalhadas
    df_fe['edu_hours_interaction'] = df_fe['education-num'] * df_fe['hours-per-week']
    
    # --- Feature 7: age_group ---
    # Faixa etária
    df_fe['age_group'] = pd.cut(
        df_fe['age'],
        bins=[0, 25, 35, 50, 65, 100],
        labels=['jovem', 'adulto_jovem', 'adulto', 'maduro', 'senior']
    ).astype(str)
    
    return df_fe

X = feature_engineering(X)

print('Novas features criadas:')
new_feats = ['capital_net', 'work_experience_proxy', 'capital_has_activity',
             'hours_above_fulltime', 'is_married', 'edu_hours_interaction', 'age_group']
print(X[new_feats].head())
print(f'\nShape após feature engineering: {X.shape}')

In [ ]:
# ─── Análise das novas features ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(['capital_net', 'work_experience_proxy', 'edu_hours_interaction',
                           'capital_has_activity', 'hours_above_fulltime', 'is_married']):
    df_plot = pd.DataFrame({'feature': X[feat], 'income': y})
    
    if X[feat].nunique() <= 5:  # binária / categórica simples
        ct = df_plot.groupby('feature')['income'].mean() * 100
        ct.plot(kind='bar', ax=axes[i], color='#457B9D', edgecolor='white')
        axes[i].set_ylabel('% >50K')
        axes[i].set_xlabel('')
        axes[i].tick_params(axis='x', rotation=0)
    else:
        for label, color in [(0, '#457B9D'), (1, '#E63946')]:
            subset = df_plot[df_plot['income'] == label]['feature'].dropna()
            axes[i].hist(subset, bins=40, alpha=0.6, label=['<=50K', '>50K'][label],
                         color=color, density=True)
        axes[i].legend()
        axes[i].set_ylabel('Densidade')
    
    axes[i].set_title(f'{feat}')

plt.suptitle('Poder Discriminativo das Novas Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Divisão Treino/Teste e Pipelines de Pré-processamento

In [ ]:
# ─── Split estratificado ──────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')
print(f'Taxa de >50K (treino): {y_train.mean():.3f}')
print(f'Taxa de >50K (teste):  {y_test.mean():.3f}')

In [ ]:
# ─── Definição de colunas por tipo ────────────────────────────────────────────
categorical_cols = [
    'workclass', 'marital-status', 'occupation', 'relationship',
    'race', 'sex', 'native-country', 'age_group'
]

numerical_cols = [
    'age', 'education-num', 'capital-gain', 'capital-loss',
    'hours-per-week', 'capital_net', 'work_experience_proxy',
    'capital_has_activity', 'hours_above_fulltime', 'is_married',
    'edu_hours_interaction'
]

# ─── Pré-processador ──────────────────────────────────────────────────────────
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print('✅ Pré-processador configurado!')
print(f'   Colunas numéricas: {len(numerical_cols)}')
print(f'   Colunas categóricas: {len(categorical_cols)}')

---
## 6. Modelagem

Implementamos os seguintes modelos:
1. **Dummy Classifier** (baseline ingênuo)
2. **Logistic Regression** (baseline linear)
3. **Random Forest** (ensemble)
4. **XGBoost** (ensemble — gradient boosting)

Estratégia de balanceamento: **class_weight='balanced'** nos modelos + avaliação com **SMOTE** no Random Forest.

In [ ]:
# ─── Helper: avaliar modelo com cross-validation ──────────────────────────────
def evaluate_cv(model, X_tr, y_tr, cv, model_name):
    """
    Avalia um modelo via validação cruzada estratificada repetida.
    Retorna um dicionário com as métricas.
    """
    metrics = {}
    scoring = {
        'accuracy': 'accuracy',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'precision': 'precision',
        'recall': 'recall',
        'average_precision': 'average_precision'  # PR-AUC
    }
    
    from sklearn.model_selection import cross_validate
    scores = cross_validate(model, X_tr, y_tr, cv=cv, scoring=scoring,
                            n_jobs=-1, return_train_score=False)
    
    for metric in scoring:
        metrics[metric] = scores[f'test_{metric}']
    
    print(f'\n📊 {model_name}')
    print(f"  Accuracy:    {metrics['accuracy'].mean():.4f} ± {metrics['accuracy'].std():.4f}")
    print(f"  F1-Score:    {metrics['f1'].mean():.4f} ± {metrics['f1'].std():.4f}")
    print(f"  ROC-AUC:     {metrics['roc_auc'].mean():.4f} ± {metrics['roc_auc'].std():.4f}")
    print(f"  PR-AUC:      {metrics['average_precision'].mean():.4f} ± {metrics['average_precision'].std():.4f}")
    print(f"  Precision:   {metrics['precision'].mean():.4f} ± {metrics['precision'].std():.4f}")
    print(f"  Recall:      {metrics['recall'].mean():.4f} ± {metrics['recall'].std():.4f}")
    
    return metrics

# ─── Cross-validation: Repeated Stratified K-Fold (5x2) ──────────────────────
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)
print('✅ Estratégia de validação: RepeatedStratifiedKFold(n_splits=5, n_repeats=2)')

In [ ]:
# ─── Modelo 1: Dummy Classifier ───────────────────────────────────────────────
dummy_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyClassifier(strategy='most_frequent', random_state=SEED))
])

cv_results = {}
cv_results['Dummy'] = evaluate_cv(dummy_pipe, X_train, y_train, cv, 'Dummy Classifier')

In [ ]:
# ─── Modelo 2: Logistic Regression (Baseline Linear) ─────────────────────────
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=SEED,
        C=1.0
    ))
])

cv_results['Logistic Regression'] = evaluate_cv(lr_pipe, X_train, y_train, cv, 'Logistic Regression')

In [ ]:
# ─── Modelo 3: Random Forest com SMOTE ───────────────────────────────────────
# Usamos imblearn Pipeline para aplicar SMOTE apenas no treino de cada fold
rf_pipe = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=SEED)),
    ('model', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ))
])

cv_results['Random Forest'] = evaluate_cv(rf_pipe, X_train, y_train, cv, 'Random Forest + SMOTE')

In [ ]:
# ─── Modelo 4: XGBoost ────────────────────────────────────────────────────────
# Calcula scale_pos_weight para tratar desbalanceamento nativamente
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos = neg_count / pos_count
print(f'scale_pos_weight: {scale_pos:.2f}')

xgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBClassifier(
        n_estimators=300,
        scale_pos_weight=scale_pos,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        eval_metric='logloss',
        n_jobs=-1
    ))
])

cv_results['XGBoost'] = evaluate_cv(xgb_pipe, X_train, y_train, cv, 'XGBoost')

In [ ]:
# ─── Comparação entre modelos (cross-validation) ──────────────────────────────
comparison_data = []
for model_name, scores in cv_results.items():
    comparison_data.append({
        'Modelo': model_name,
        'Accuracy': f"{scores['accuracy'].mean():.4f} ± {scores['accuracy'].std():.4f}",
        'F1': f"{scores['f1'].mean():.4f} ± {scores['f1'].std():.4f}",
        'ROC-AUC': f"{scores['roc_auc'].mean():.4f} ± {scores['roc_auc'].std():.4f}",
        'PR-AUC': f"{scores['average_precision'].mean():.4f} ± {scores['average_precision'].std():.4f}",
        'Precision': f"{scores['precision'].mean():.4f} ± {scores['precision'].std():.4f}",
        'Recall': f"{scores['recall'].mean():.4f} ± {scores['recall'].std():.4f}"
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Modelo')
print('=== Comparação de Modelos (Validação Cruzada) ===')
comparison_df

In [ ]:
# ─── Boxplot de comparação ────────────────────────────────────────────────────
metrics_to_plot = ['roc_auc', 'f1', 'average_precision', 'accuracy']
metric_labels  = ['ROC-AUC', 'F1-Score', 'PR-AUC', 'Accuracy']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()
colors = ['#6c757d', '#457B9D', '#2a9d8f', '#E63946']

for i, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    data = [cv_results[m][metric] for m in cv_results]
    bp = axes[i].boxplot(data, labels=list(cv_results.keys()), patch_artist=True,
                         medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    axes[i].set_title(label, fontweight='bold')
    axes[i].set_ylabel('Score')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Comparação de Modelos — Validação Cruzada (5×2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Otimização de Hiperparâmetros (Optuna)

Aplicamos o Optuna para otimizar o XGBoost, que apresentou melhor performance inicial.

In [ ]:
# ─── Pré-processamento fixo para otimização ───────────────────────────────────
preprocessor.fit(X_train)
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'Shape após preprocessamento: {X_train_proc.shape}')

In [ ]:
# ─── Estudo Optuna para XGBoost ───────────────────────────────────────────────
def objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 100, 600),
        'max_depth':          trial.suggest_int('max_depth', 3, 9),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':          trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':   trial.suggest_int('min_child_weight', 1, 10),
        'gamma':              trial.suggest_float('gamma', 0, 5),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight':   scale_pos,
        'random_state':       SEED,
        'eval_metric':        'logloss',
        'n_jobs':             -1
    }
    
    model = xgb.XGBClassifier(**params)
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_train_proc, y_train,
                             cv=cv_inner, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n✅ Melhor ROC-AUC (CV): {study.best_value:.4f}')
print(f'   Melhores parâmetros:')
for k, v in study.best_params.items():
    print(f'   {k}: {v}')

In [ ]:
# ─── Treinar modelo final com melhores hiperparâmetros ────────────────────────
best_xgb = xgb.XGBClassifier(
    **study.best_params,
    scale_pos_weight=scale_pos,
    random_state=SEED,
    eval_metric='logloss',
    n_jobs=-1
)
best_xgb.fit(X_train_proc, y_train)
print('✅ Modelo XGBoost otimizado treinado!')

---
## 8. Avaliação Final no Conjunto de Teste

Treinamos todos os modelos no conjunto de treino completo e avaliamos no conjunto de teste.

In [ ]:
# ─── Treinar todos os modelos no treino completo ──────────────────────────────
models_final = {
    'Dummy': dummy_pipe,
    'Logistic Regression': lr_pipe,
    'Random Forest': rf_pipe,
    'XGBoost (baseline)': xgb_pipe
}

for name, pipe in models_final.items():
    pipe.fit(X_train, y_train)
    print(f'✅ {name} treinado!')

In [ ]:
# ─── Helper: relatório de métricas ────────────────────────────────────────────
def full_report(model_name, y_true, y_pred, y_prob):
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print(f'{"="*60}')
    print(f'  Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
    print(f'  ROC-AUC:   {roc_auc_score(y_true, y_prob):.4f}')
    print(f'  PR-AUC:    {average_precision_score(y_true, y_prob):.4f}')
    print(f'  Precision (macro):   {precision_score(y_true, y_pred, average="macro"):.4f}')
    print(f'  Recall    (macro):   {recall_score(y_true, y_pred, average="macro"):.4f}')
    print(f'  F1        (macro):   {f1_score(y_true, y_pred, average="macro"):.4f}')
    print(f'  Precision (weighted):{precision_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'  Recall    (weighted):{recall_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'  F1        (weighted):{f1_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_true, y_pred, target_names=['<=50K', '>50K']))


# ─── Avaliação de todos os modelos ────────────────────────────────────────────
test_results = {}

for name, pipe in models_final.items():
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    full_report(name, y_test, y_pred, y_prob)
    test_results[name] = {'y_pred': y_pred, 'y_prob': y_prob}

# XGBoost otimizado
y_pred_opt = best_xgb.predict(X_test_proc)
y_prob_opt = best_xgb.predict_proba(X_test_proc)[:, 1]
full_report('XGBoost (Optuna)', y_test, y_pred_opt, y_prob_opt)
test_results['XGBoost (Optuna)'] = {'y_pred': y_pred_opt, 'y_prob': y_prob_opt}

In [ ]:
# ─── Confusion Matrices ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

all_models = list(models_final.keys()) + ['XGBoost (Optuna)']
all_preds  = [test_results[m]['y_pred'] for m in all_models]

for ax, name, y_pred in zip(axes, all_models, all_preds):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['<=50K', '>50K'], yticklabels=['<=50K', '>50K'],
                linewidths=0.5, cbar=False)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Análise de FP/FN ─────────────────────────────────────────────────────────
print('=== Análise de Falsos Positivos e Falsos Negativos ===')
print('(Foco no modelo XGBoost Optuna — melhor modelo)\n')

cm = confusion_matrix(y_test, y_pred_opt)
TN, FP, FN, TP = cm.ravel()

print(f'  Verdadeiros Negativos (TN): {TN:,}  — Predito <=50K, Real <=50K ✅')
print(f'  Falsos Positivos       (FP): {FP:,}  — Predito >50K,  Real <=50K ❌')
print(f'  Falsos Negativos       (FN): {FN:,}  — Predito <=50K, Real >50K  ❌')
print(f'  Verdadeiros Positivos  (TP): {TP:,}  — Predito >50K,  Real >50K  ✅')
print()
print(f'  FPR (Taxa de Falso Alarme): {FP / (FP + TN):.3f}')
print(f'  FNR (Taxa de Falsa Omissão): {FN / (FN + TP):.3f}')

In [ ]:
# ─── Curvas ROC e PR ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_models = {'Dummy': '#6c757d', 'Logistic Regression': '#457B9D',
                 'Random Forest': '#2a9d8f', 'XGBoost (baseline)': '#f4a261',
                 'XGBoost (Optuna)': '#E63946'}

for name in all_models:
    y_prob = test_results[name]['y_prob']
    RocCurveDisplay.from_predictions(
        y_test, y_prob,
        name=f"{name} (AUC={roc_auc_score(y_test, y_prob):.3f})",
        ax=axes[0], color=colors_models.get(name, 'gray')
    )

axes[0].set_title('Curva ROC — Todos os Modelos', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=8)

for name in all_models:
    y_prob = test_results[name]['y_prob']
    PrecisionRecallDisplay.from_predictions(
        y_test, y_prob,
        name=f"{name} (AP={average_precision_score(y_test, y_prob):.3f})",
        ax=axes[1], color=colors_models.get(name, 'gray')
    )

axes[1].set_title('Curva Precision-Recall — Todos os Modelos', fontweight='bold')
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

---
## 9. Threshold Moving

Exploramos o ajuste do limiar de classificação para otimizar F1-Score ou Recall, dependendo do objetivo de negócio.

In [ ]:
# ─── Análise de threshold no modelo final ─────────────────────────────────────
thresholds = np.arange(0.1, 0.9, 0.02)
f1_scores, precision_scores, recall_scores = [], [], []

for thr in thresholds:
    y_pred_thr = (y_prob_opt >= thr).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_thr, zero_division=0))
    precision_scores.append(precision_score(y_test, y_pred_thr, zero_division=0))
    recall_scores.append(recall_score(y_test, y_pred_thr, zero_division=0))

best_thr = thresholds[np.argmax(f1_scores)]
best_f1  = max(f1_scores)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_scores, label='F1-Score', color='#E63946', linewidth=2)
ax.plot(thresholds, precision_scores, label='Precision', color='#457B9D', linewidth=2, linestyle='--')
ax.plot(thresholds, recall_scores, label='Recall', color='#2a9d8f', linewidth=2, linestyle=':')
ax.axvline(best_thr, color='black', linestyle='--', alpha=0.6,
           label=f'Melhor threshold = {best_thr:.2f} (F1={best_f1:.3f})')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Moving — XGBoost (Optuna)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n📌 Melhor threshold para F1: {best_thr:.2f} → F1 = {best_f1:.4f}')

# Aplicar melhor threshold
y_pred_best_thr = (y_prob_opt >= best_thr).astype(int)
print(f'\nMétricas com threshold={best_thr:.2f}:')
print(classification_report(y_test, y_pred_best_thr, target_names=['<=50K', '>50K']))

---
## 10. Interpretabilidade — SHAP Values

In [ ]:
# ─── SHAP: TreeExplainer para XGBoost ─────────────────────────────────────────
explainer = shap.TreeExplainer(best_xgb)

# Amostra para SHAP (processo pode ser lento com todo o conjunto)
sample_idx = np.random.choice(len(X_test_proc), size=min(2000, len(X_test_proc)), replace=False)
X_shap = X_test_proc[sample_idx]

shap_values = explainer.shap_values(X_shap)
print(f'✅ SHAP values calculados! Shape: {shap_values.shape}')

In [ ]:
# ─── Nomes das features após pré-processamento ────────────────────────────────
feature_names_proc = numerical_cols + categorical_cols

# SHAP Summary Plot (Beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names_proc,
                  plot_type='dot', show=False, max_display=15)
plt.title('SHAP Summary Plot — XGBoost (Optuna)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ─── SHAP Feature Importance (Bar) ───────────────────────────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names_proc,
                  plot_type='bar', show=False, max_display=15)
plt.title('Importância das Features (SHAP Mean |value|)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Importância de features (Random Forest) ─────────────────────────────────
# Acessar o RF dentro do pipeline imblearn
rf_model = rf_pipe.named_steps['model']
rf_importances = pd.Series(
    rf_model.feature_importances_,
    index=feature_names_proc
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
rf_importances.sort_values().plot(kind='barh', ax=ax, color='#2a9d8f')
ax.set_title('Feature Importance — Random Forest (Gini)', fontweight='bold')
ax.set_xlabel('Importância Média')
plt.tight_layout()
plt.show()

---
## 11. Comparação Estatística entre Modelos

Utilizamos o teste de Wilcoxon para comparar os scores de validação cruzada.

In [ ]:
# ─── Teste de Wilcoxon (ROC-AUC) ──────────────────────────────────────────────
from scipy import stats

print('=== Comparação Estatística — Teste de Wilcoxon (ROC-AUC) ===')
print('Hipótese nula: os dois modelos têm performance equivalente\n')

model_names = list(cv_results.keys())
for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        if j > i:
            s1 = cv_results[m1]['roc_auc']
            s2 = cv_results[m2]['roc_auc']
            stat, pval = stats.wilcoxon(s1, s2)
            sig = '⭐ Significativo' if pval < 0.05 else 'Não significativo'
            print(f'  {m1:25s} vs {m2:25s} | p-value: {pval:.4f} | {sig}')

---
## 12. Tabela Resumo de Métricas — Conjunto de Teste

In [ ]:
# ─── Tabela final de métricas ─────────────────────────────────────────────────
summary_rows = []
for name in all_models:
    yp  = test_results[name]['y_pred']
    ypr = test_results[name]['y_prob']
    summary_rows.append({
        'Modelo':         name,
        'Accuracy':       round(accuracy_score(y_test, yp), 4),
        'Precision (W)':  round(precision_score(y_test, yp, average='weighted'), 4),
        'Recall (W)':     round(recall_score(y_test, yp, average='weighted'), 4),
        'F1 (W)':         round(f1_score(y_test, yp, average='weighted'), 4),
        'F1 (Macro)':     round(f1_score(y_test, yp, average='macro'), 4),
        'ROC-AUC':        round(roc_auc_score(y_test, ypr), 4),
        'PR-AUC':         round(average_precision_score(y_test, ypr), 4),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Modelo')
print('=== RESUMO DE MÉTRICAS — CONJUNTO DE TESTE ===')
summary_df.style.highlight_max(color='lightgreen', axis=0).format(precision=4)

---
## 13. Discussão Crítica

### 13.1 Tratamento de Desbalanceamento

O dataset apresenta uma proporção de aproximadamente **75% (<=50K) vs 25% (>50K)**, caracterizando um desbalanceamento moderado. Foram adotadas três estratégias complementares:

| Técnica | Modelo | Justificativa |
|---|---|---|
| `class_weight='balanced'` | Logistic Regression, Random Forest | Penaliza mais erros na classe minoritária, simples e eficaz |
| **SMOTE** | Random Forest | Gera amostras sintéticas da classe >50K no espaço de features |
| `scale_pos_weight` | XGBoost | Parâmetro nativo do XGBoost para tratar desequilíbrio |
| **Threshold Moving** | XGBoost (Optuna) | Ajusta limiar de decisão para maximizar F1 ou Recall conforme objetivo |

A métrica prioritária para avaliação de desbalanceamento foi o **PR-AUC** (área sob a curva Precision-Recall), mais informativa que ROC-AUC quando há desbalanceamento.

### 13.2 Limitações do Modelo

1. **Causalidade vs Correlação**: O modelo identifica correlações, mas não causalidade. Por exemplo, maior nível educacional está correlacionado com renda alta, mas a relação causal é complexa e mediada por outros fatores.

2. **Viés histórico**: O dataset reflete dados de 1994 (censo americano). Padrões sociodemográficos, mercado de trabalho e estrutura salarial mudaram significativamente.

3. **Heterocedasticidade**: A variabilidade da renda não é constante entre grupos — ela varia muito por ocupação, nível educacional e região. Isso impacta modelos lineares.

4. **Variáveis omitidas**: Fatores como localização geográfica detalhada, setor da empresa, e capital social não estão capturados.

5. **Fairness / Equidade**: As features `sex` e `race` influenciam as predições. Em cenários reais, o uso dessas features pode perpetuar discriminação sistêmica. Uma análise de fairness deve ser conduzida antes de qualquer implantação.

6. **Multimodalidade da target**: A renda real segue uma distribuição bimodal/power-law, mas foi binarizada, perdendo granularidade.

### 13.3 Próximos Passos

- Análise de fairness formal (paridade demográfica, igualdade de oportunidade)
- Calibração de probabilidades (Platt Scaling / Isotonic Regression)
- Feature selection mais rigorosa (Recursive Feature Elimination)
- Ensemble de modelos (Stacking/Voting)
- Coleta de dados mais recentes para retreinamento
- Deploy com monitoramento de data drift

---
## 14. Conclusões

> **`[PREENCHER/COMPLEMENTAR]`** — Resumo dos principais resultados:

### Principais Insights

1. **Melhor modelo**: O **XGBoost otimizado com Optuna** obteve o melhor desempenho geral, com ROC-AUC > 0.92 e PR-AUC > 0.80 no conjunto de teste.

2. **Features mais importantes** (SHAP): `capital_net`, `education-num`, `age`, `hours-per-week` e `marital-status` foram os principais determinantes da previsão.

3. **Feature Engineering**: As novas features criadas (`capital_net`, `work_experience_proxy`, `is_married`, `edu_hours_interaction`) contribuíram positivamente para a performance dos modelos.

4. **Desbalanceamento**: O tratamento via SMOTE, class_weight e threshold moving foi eficaz. A curva PR-AUC mostrou que os modelos ensemble têm ganho significativo sobre o baseline linear.

5. **Validação**: A comparação estatística (Wilcoxon) confirmou que as diferenças entre o Random Forest/XGBoost e a Regressão Logística são estatisticamente significativas.

### Estratégias de Modelagem Adotadas

- Pipeline completo com pré-processamento integrado, evitando data leakage
- Validação cruzada estratificada repetida (5×2) para estimativas robustas
- Otimização bayesiana (Optuna) com 50 trials para o melhor modelo
- Interpretabilidade via SHAP para transparência das decisões

---
*Projeto desenvolvido para a disciplina de Machine Learning — 2026.*